In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Original Subset from images_001

In [ ]:
import pandas as pd
import os
import shutil

#Load CSV with image metadata and labels
csv_path = "/content/drive/MyDrive/NIH_ChestXray/Data_Entry_2017.csv"
df = pd.read_csv(csv_path)

# Define target classes
# keep 4 classes only for this project
target_classes = ["No Finding", "Pneumonia", "Effusion", "Cardiomegaly"]

# Keep only rows where the label matches one of the target classes exactly
df_filtered = df[df["Finding Labels"].isin(target_classes)]

print("Original dataset size:", len(df))
print("Filtered dataset size:", len(df_filtered))

Original dataset size: 112120
Filtered dataset size: 65731


In [ ]:
# Split multi-label entries into separate labels
all_labels = []
for entry in df["Finding Labels"]:
    labels = entry.split('|')  # labels are separated by "|"
    all_labels.extend(labels)

# Count frequency
label_counts = pd.Series(all_labels).value_counts()

# Convert to percentage
label_percent = (label_counts / len(df)) * 100

# Combine into a single DataFrame
summary = pd.DataFrame({
    "Count": label_counts,
    "Percentage": label_percent.round(2)
})

print(summary)

                    Count  Percentage
No Finding          60361       53.84
Infiltration        19894       17.74
Effusion            13317       11.88
Atelectasis         11559       10.31
Nodule               6331        5.65
Mass                 5782        5.16
Pneumothorax         5302        4.73
Consolidation        4667        4.16
Pleural_Thickening   3385        3.02
Cardiomegaly         2776        2.48
Emphysema            2516        2.24
Edema                2303        2.05
Fibrosis             1686        1.50
Pneumonia            1431        1.28
Hernia                227        0.20


In [ ]:
# Create output directories for each class
output_dir = "/content/drive/MyDrive/NIH_ChestXray_subset"
os.makedirs(output_dir, exist_ok=True)

for cls in target_classes:
    os.makedirs(os.path.join(output_dir, cls), exist_ok=True)

# Copy selected images into class-specific folders
images_dir = "/content/drive/MyDrive/NIH_ChestXray/images"

for _, row in df_filtered.iterrows():
    fname = row["Image Index"]
    label = row["Finding Labels"]

    src = os.path.join(images_dir, fname)
    dst = os.path.join(output_dir, label, fname)

    if os.path.exists(src):
        shutil.copy(src, dst)

print("Subset dataset created successfully!")


Subset dataset created successfully!


In [ ]:
counts = {}
for cls in os.listdir(output_dir):
    class_path = os.path.join(output_dir, cls)
    if os.path.isdir(class_path):
        num_files = len([f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        counts[cls] = num_files
        print(f"{cls}: {num_files} images")

No Finding: 5999 images
Pneumonia: 34 images
Effusion: 309 images
Cardiomegaly: 91 images


In [ ]:
total = sum(counts.values())

for cls, num in counts.items():
    ratio = num / total * 100
    print(f"{cls}: {num} images ({ratio:.2f}%)")

print(f"\nTotal images: {total}")

No Finding: 5999 images (93.25%)
Pneumonia: 34 images (0.53%)
Effusion: 309 images (4.80%)
Cardiomegaly: 91 images (1.41%)

Total images: 6433


In [ ]:
# Create train/val/test split for the original subset
import random

def create_split_original(source_dir, dest_dir, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    """
    Split dataset into train/val/test sets
    Default: 70% train, 15% val, 15% test
    """
    if os.path.exists(dest_dir):
        print(f"Directory {dest_dir} already exists. Skipping split creation.")
        return
    
    random.seed(42)
    os.makedirs(dest_dir, exist_ok=True)
    
    # Create split directories
    for split in ['train', 'val', 'test']:
        os.makedirs(f"{dest_dir}/{split}", exist_ok=True)
    
    # Get all class directories
    class_dirs = [d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))]
    
    for class_name in class_dirs:
        class_path = os.path.join(source_dir, class_name)
        image_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        
        # Shuffle images
        random.shuffle(image_files)
        n_total = len(image_files)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)
        
        # Split files
        splits = {
            'train': image_files[:n_train],
            'val': image_files[n_train:n_train + n_val],
            'test': image_files[n_train + n_val:]
        }
        
        # Copy files to respective directories
        for split, files in splits.items():
            split_class_dir = f"{dest_dir}/{split}/{class_name}"
            os.makedirs(split_class_dir, exist_ok=True)
            for file in files:
                shutil.copy2(os.path.join(class_path, file), os.path.join(split_class_dir, file))
        
        print(f"{class_name}: Train={len(splits['train'])}, Val={len(splits['val'])}, Test={len(splits['test'])}")

# Create the split for original subset
source_data_dir = "/content/drive/MyDrive/NIH_ChestXray_subset"
split_data_dir = "/content/drive/MyDrive/NIH_ChestXray_subset_split"

create_split_original(source_data_dir, split_data_dir)
print("\nData split completed for original subset!")

In [ ]:
# Verify the split - count images in train/val/test for original subset
split_dir_original = "/content/drive/MyDrive/NIH_ChestXray_subset_split"

for split in ['train', 'val', 'test']:
    split_path = os.path.join(split_dir_original, split)
    if os.path.exists(split_path):
        total_images = 0
        print(f"\n{split.upper()}:")
        for class_name in os.listdir(split_path):
            class_path = os.path.join(split_path, class_name)
            if os.path.isdir(class_path):
                num_images = len([f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
                total_images += num_images
                print(f"  {class_name}: {num_images} images")
        print(f"  Total: {total_images} images")

# Summary
print("\n" + "="*50)
print("SUMMARY - ORIGINAL SUBSET")
print("="*50)
print(f"Source directory: /content/drive/MyDrive/NIH_ChestXray/images")
print(f"Subset directory: {source_data_dir}")
print(f"Split directory: {split_data_dir}")
print(f"Train/Val/Test ratio: 70%/15%/15%")

# New Subset from images_010

In [ ]:
# Create subset from images_010 directory
import pandas as pd
import os
import shutil

# Load CSV with image metadata and labels
csv_path = "/content/drive/MyDrive/NIH_ChestXray/Data_Entry_2017.csv"
df_new = pd.read_csv(csv_path)

# Define target classes (same as before)
target_classes = ["No Finding", "Pneumonia", "Effusion", "Cardiomegaly"]

# Keep only rows where the label matches one of the target classes exactly
df_filtered_new = df_new[df_new["Finding Labels"].isin(target_classes)]

print("Original dataset size:", len(df_new))
print("Filtered dataset size:", len(df_filtered_new))

# Create output directories for each class
output_dir_new = "/content/drive/MyDrive/NIH_ChestXray_subset_010"
os.makedirs(output_dir_new, exist_ok=True)

for cls in target_classes:
    os.makedirs(os.path.join(output_dir_new, cls), exist_ok=True)

# Copy selected images from images_010 into class-specific folders
images_dir_new = "/content/drive/MyDrive/images_010"

copied_count = 0
not_found_count = 0

for _, row in df_filtered_new.iterrows():
    fname = row["Image Index"]
    label = row["Finding Labels"]
    
    src = os.path.join(images_dir_new, fname)
    dst = os.path.join(output_dir_new, label, fname)
    
    if os.path.exists(src):
        shutil.copy(src, dst)
        copied_count += 1
    else:
        not_found_count += 1

print(f"\nSubset dataset created successfully!")
print(f"Copied: {copied_count} images")
print(f"Not found: {not_found_count} images")

In [ ]:
# Check the number of images in each class
counts_new = {}
for cls in os.listdir(output_dir_new):
    class_path = os.path.join(output_dir_new, cls)
    if os.path.isdir(class_path):
        num_files = len([f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        counts_new[cls] = num_files
        print(f"{cls}: {num_files} images")

total_new = sum(counts_new.values())

print(f"\nClass distribution:")
for cls, num in counts_new.items():
    ratio = num / total_new * 100
    print(f"{cls}: {num} images ({ratio:.2f}%)")

print(f"\nTotal images: {total_new}")

In [ ]:
# Create train/val/test split for the new subset
import random
from sklearn.model_selection import train_test_split

def create_split_010(source_dir, dest_dir, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15):
    """
    Split dataset into train/val/test sets
    Default: 70% train, 15% val, 15% test
    """
    if os.path.exists(dest_dir):
        print(f"Directory {dest_dir} already exists. Skipping split creation.")
        return
    
    random.seed(42)
    os.makedirs(dest_dir, exist_ok=True)
    
    # Create split directories
    for split in ['train', 'val', 'test']:
        os.makedirs(f"{dest_dir}/{split}", exist_ok=True)
    
    # Get all class directories
    class_dirs = [d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))]
    
    for class_name in class_dirs:
        class_path = os.path.join(source_dir, class_name)
        image_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        
        # Shuffle images
        random.shuffle(image_files)
        n_total = len(image_files)
        n_train = int(n_total * train_ratio)
        n_val = int(n_total * val_ratio)
        
        # Split files
        splits = {
            'train': image_files[:n_train],
            'val': image_files[n_train:n_train + n_val],
            'test': image_files[n_train + n_val:]
        }
        
        # Copy files to respective directories
        for split, files in splits.items():
            split_class_dir = f"{dest_dir}/{split}/{class_name}"
            os.makedirs(split_class_dir, exist_ok=True)
            for file in files:
                shutil.copy2(os.path.join(class_path, file), os.path.join(split_class_dir, file))
        
        print(f"{class_name}: Train={len(splits['train'])}, Val={len(splits['val'])}, Test={len(splits['test'])}")

# Create the split
source_data_dir_new = "/content/drive/MyDrive/NIH_ChestXray_subset_010"
split_data_dir_new = "/content/drive/MyDrive/NIH_ChestXray_subset_010_split"

create_split_010(source_data_dir_new, split_data_dir_new)
print("\nData split completed for images_010 subset!")

In [ ]:
# Verify the split - count images in train/val/test
split_dir = "/content/drive/MyDrive/NIH_ChestXray_subset_010_split"

for split in ['train', 'val', 'test']:
    split_path = os.path.join(split_dir, split)
    if os.path.exists(split_path):
        total_images = 0
        print(f"\n{split.upper()}:")
        for class_name in os.listdir(split_path):
            class_path = os.path.join(split_path, class_name)
            if os.path.isdir(class_path):
                num_images = len([f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
                total_images += num_images
                print(f"  {class_name}: {num_images} images")
        print(f"  Total: {total_images} images")

# Summary
print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"Source directory: /content/drive/images_010")
print(f"Subset directory: {source_data_dir_new}")
print(f"Split directory: {split_data_dir_new}")
print(f"Train/Val/Test ratio: 70%/15%/15%")

# Comparison Between Two Subsets

In [ ]:
# Compare both datasets side by side
import pandas as pd

def count_split_images(split_dir, dataset_name):
    """Count images in each split and return as dictionary"""
    results = {}
    for split in ['train', 'val', 'test']:
        split_path = os.path.join(split_dir, split)
        if os.path.exists(split_path):
            class_counts = {}
            total = 0
            for class_name in os.listdir(split_path):
                class_path = os.path.join(split_path, class_name)
                if os.path.isdir(class_path):
                    num_images = len([f for f in os.listdir(class_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
                    class_counts[class_name] = num_images
                    total += num_images
            results[split] = {'classes': class_counts, 'total': total}
    return results

# Count images in both datasets
original_counts = count_split_images("/content/drive/MyDrive/NIH_ChestXray_subset_split", "Original")
new_counts = count_split_images("/content/drive/MyDrive/NIH_ChestXray_subset_010_split", "images_010")

# Display comparison
print("="*80)
print("COMPARISON: Original (images) vs images_010")
print("="*80)

for split in ['train', 'val', 'test']:
    print(f"\n{split.upper()}:")
    print("-" * 80)
    
    # Get all unique class names from both datasets
    all_classes = set(list(original_counts.get(split, {}).get('classes', {}).keys()) + 
                     list(new_counts.get(split, {}).get('classes', {}).keys()))
    
    print(f"{'Class':<20} {'Original':<15} {'images_010':<15} {'Difference':<15}")
    print("-" * 80)
    
    for cls in sorted(all_classes):
        orig_count = original_counts.get(split, {}).get('classes', {}).get(cls, 0)
        new_count = new_counts.get(split, {}).get('classes', {}).get(cls, 0)
        diff = new_count - orig_count
        print(f"{cls:<20} {orig_count:<15} {new_count:<15} {diff:+<15}")
    
    print("-" * 80)
    orig_total = original_counts.get(split, {}).get('total', 0)
    new_total = new_counts.get(split, {}).get('total', 0)
    total_diff = new_total - orig_total
    print(f"{'TOTAL':<20} {orig_total:<15} {new_total:<15} {total_diff:+<15}")

print("\n" + "="*80)
print("OVERALL SUMMARY")
print("="*80)

# Calculate totals across all splits
orig_grand_total = sum(original_counts.get(s, {}).get('total', 0) for s in ['train', 'val', 'test'])
new_grand_total = sum(new_counts.get(s, {}).get('total', 0) for s in ['train', 'val', 'test'])
grand_diff = new_grand_total - orig_grand_total

print(f"Original dataset total: {orig_grand_total} images")
print(f"images_010 dataset total: {new_grand_total} images")
print(f"Difference: {grand_diff:+} images ({(grand_diff/orig_grand_total*100 if orig_grand_total > 0 else 0):.2f}%)")


# Reduce images_010 Dataset to Match Original Dataset Size

In [ ]:
import random
import shutil
import os

def reduce_dataset_to_match(source_split_dir, target_split_dir, reference_counts):
    """
    Reduce the number of images in source_split_dir to match the reference_counts
    by randomly removing excess images
    
    Args:
        source_split_dir: Directory to reduce (e.g., NIH_ChestXray_subset_010_split)
        target_split_dir: New directory to save reduced dataset
        reference_counts: Dictionary with target counts from original dataset
    """
    random.seed(42)  # For reproducibility
    
    # Create target directory
    os.makedirs(target_split_dir, exist_ok=True)
    
    print("="*80)
    print("REDUCING DATASET TO MATCH ORIGINAL SIZE")
    print("="*80)
    
    for split in ['train', 'val', 'test']:
        print(f"\n{split.upper()}:")
        print("-" * 80)
        
        source_split_path = os.path.join(source_split_dir, split)
        target_split_path = os.path.join(target_split_dir, split)
        os.makedirs(target_split_path, exist_ok=True)
        
        ref_classes = reference_counts.get(split, {}).get('classes', {})
        
        for class_name in os.listdir(source_split_path):
            source_class_path = os.path.join(source_split_path, class_name)
            
            if not os.path.isdir(source_class_path):
                continue
            
            target_class_path = os.path.join(target_split_path, class_name)
            os.makedirs(target_class_path, exist_ok=True)
            
            # Get all images in source
            all_images = [f for f in os.listdir(source_class_path) 
                         if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            
            current_count = len(all_images)
            target_count = ref_classes.get(class_name, current_count)
            
            # If we have more images than needed, randomly sample
            if current_count > target_count:
                selected_images = random.sample(all_images, target_count)
                action = f"Reduced from {current_count} to {target_count}"
            else:
                selected_images = all_images
                action = f"Kept all {current_count} images (target: {target_count})"
            
            # Copy selected images
            for img in selected_images:
                src = os.path.join(source_class_path, img)
                dst = os.path.join(target_class_path, img)
                shutil.copy2(src, dst)
            
            print(f"  {class_name}: {action}")
    
    print("\n" + "="*80)
    print("DATASET REDUCTION COMPLETED!")
    print("="*80)

# Get reference counts from original dataset
original_counts = count_split_images("/content/drive/MyDrive/NIH_ChestXray_subset_split", "Original")

# Reduce the images_010 dataset
source_010_split = "/content/drive/MyDrive/NIH_ChestXray_subset_010_split"
reduced_010_split = "/content/drive/MyDrive/NIH_ChestXray_subset_010_split_reduced"

reduce_dataset_to_match(source_010_split, reduced_010_split, original_counts)


In [ ]:
# Verify the reduced dataset
reduced_counts = count_split_images("/content/drive/MyDrive/NIH_ChestXray_subset_010_split_reduced", "Reduced images_010")

print("="*80)
print("VERIFICATION: Original vs Reduced images_010")
print("="*80)

for split in ['train', 'val', 'test']:
    print(f"\n{split.upper()}:")
    print("-" * 80)
    
    all_classes = set(list(original_counts.get(split, {}).get('classes', {}).keys()) + 
                     list(reduced_counts.get(split, {}).get('classes', {}).keys()))
    
    print(f"{'Class':<20} {'Original':<15} {'Reduced 010':<15} {'Match?':<15}")
    print("-" * 80)
    
    for cls in sorted(all_classes):
        orig_count = original_counts.get(split, {}).get('classes', {}).get(cls, 0)
        reduced_count = reduced_counts.get(split, {}).get('classes', {}).get(cls, 0)
        match = "✓ YES" if orig_count == reduced_count else "✗ NO"
        print(f"{cls:<20} {orig_count:<15} {reduced_count:<15} {match:<15}")
    
    print("-" * 80)
    orig_total = original_counts.get(split, {}).get('total', 0)
    reduced_total = reduced_counts.get(split, {}).get('total', 0)
    total_match = "✓ YES" if orig_total == reduced_total else "✗ NO"
    print(f"{'TOTAL':<20} {orig_total:<15} {reduced_total:<15} {total_match:<15}")

print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

orig_grand_total = sum(original_counts.get(s, {}).get('total', 0) for s in ['train', 'val', 'test'])
reduced_grand_total = sum(reduced_counts.get(s, {}).get('total', 0) for s in ['train', 'val', 'test'])

print(f"Original dataset total: {orig_grand_total} images")
print(f"Reduced images_010 dataset total: {reduced_grand_total} images")
print(f"Match: {'✓ YES' if orig_grand_total == reduced_grand_total else '✗ NO'}")
print(f"\nReduced dataset location: /content/drive/MyDrive/NIH_ChestXray_subset_010_split_reduced")
